# Пробный запуск MoGe2-Aerial на кадрах дрона

Ноутбук запускает готовую UAV-адаптированную модель глубины на небольшой выборке кадров. Он сохраняет карты глубины, визуализации и таблицу для ориентировочного сравнения с `dist`.

`dist` — целевая метка. Центральная область кадра используется только как пробное сопоставление: положение лидара относительно кадра пока не откалибровано, поэтому полученная ошибка не является оценкой точности модели.

До запуска подключите GPU и Internet в настройках Kaggle и добавьте как Input файл `kaggle_aerial_sample.zip`.

In [ ]:
import os, sys, subprocess, pathlib, glob, zipfile, shutil, json, time
from pathlib import Path
import torch, requests
print("Python:", sys.version)
print("PyTorch:", torch.__version__, "CUDA runtime:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Kaggle запустил сессию без GPU. Включите GPU в Settings → Accelerator и перезапустите.")
print("GPU:", torch.cuda.get_device_name(0))
requests.get("https://github.com", timeout=30).raise_for_status()
print("Internet: доступен")

In [ ]:
# Установка исходного кода модели и зависимостей
repo_dir = Path('/kaggle/working/AerialMetric')
if not repo_dir.exists():
    subprocess.run(['git','clone','https://github.com/kuieless/AerialMetric.git',str(repo_dir)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-r',str(repo_dir/'requirements.txt')],check=True)
subprocess.run([sys.executable,'-m','pip','install','-e',str(repo_dir/'MoGe')],check=True)
# У Kaggle встречается старая необязательная torchao, несовместимая с PEFT.
subprocess.run([sys.executable,'-m','pip','uninstall','-y','torchao'],check=False)

In [ ]:
# Kaggle может показать исходный ZIP или уже распакованные файлы датасета
input_root=Path('/kaggle/input')
archives=list(input_root.rglob('kaggle_aerial_sample.zip'))
if archives:
    data_dir=Path('/kaggle/working/aerial_sample')
    if data_dir.exists(): shutil.rmtree(data_dir)
    data_dir.mkdir(parents=True)
    with zipfile.ZipFile(archives[0]) as z: z.extractall(data_dir)
else:
    label_files=[p for p in input_root.rglob('labels.csv') if (p.parent/'images').is_dir()]
    if not label_files:
        raise FileNotFoundError(f'Не нашёл ZIP или папку с labels.csv и images/ в {input_root}. Содержимое: {list(input_root.iterdir())}')
    data_dir=label_files[0].parent
image_dir=data_dir/'images'
images=list(image_dir.glob('*'))
print('Кадров:', len(images))
print('Папка с данными:', data_dir)
if not images: raise FileNotFoundError(f'В папке нет изображений: {image_dir}')

In [ ]:
# Скачать только папку с весами модели; большой датасет AerialMetric не загружается
from huggingface_hub import snapshot_download
weights_dir = Path('/kaggle/working/aerialmetric_weights')
snapshot_download(
    repo_id='Kuiee/AerialMetric-ECCV2026',
    repo_type='dataset',
    allow_patterns='weights/*',
    local_dir=str(weights_dir)
)
checkpoints = list(weights_dir.rglob('*.pt'))
print('Найденные веса:', checkpoints)
aerial_weights = [p for p in checkpoints if 'aerial' in p.name.lower()]
if not aerial_weights:
    raise FileNotFoundError('В папке weights/ не найден .pt с названием, содержащим aerial. Посмотрите список выше.')
checkpoint = aerial_weights[0]
print('Используем:', checkpoint)

In [ ]:
# Запуск. Для первого просмотра ограничиваем длинную сторону 896 px и уровень качества 7/9.
output_dir = Path('/kaggle/working/moge2_aerial_output')
cmd = [sys.executable, str(repo_dir/'demo_infer.py'),
       '--input', str(image_dir), '--model', 'lora',
       '--checkpoint', str(checkpoint), '--output', str(output_dir),
       '--resize', '896', '--resolution_level', '7', '--save_components']
print(' '.join(map(str,cmd)))
subprocess.run(cmd, cwd=repo_dir, check=True)
print('Результаты:', output_dir)

In [ ]:
# Посмотреть исходный кадр и предсказанную карту глубины для нескольких примеров
from PIL import Image
import matplotlib.pyplot as plt
outputs = sorted(output_dir.glob('*_depth_annotated.png'))
print('Получено карт глубины:', len(outputs))
fig, axes = plt.subplots(min(3,len(outputs)), 2, figsize=(14, 5*min(3,len(outputs))))
if len(outputs)==1: axes=[axes]
for ax, out in zip(axes, outputs[:3]):
    stem=out.name.replace('_depth_annotated.png','')
    original=image_dir/(stem+'.jpeg')
    if not original.exists(): original=image_dir/(stem+'.jpg')
    ax[0].imshow(Image.open(original)); ax[0].set_title(stem); ax[0].axis('off')
    ax[1].imshow(Image.open(out)); ax[1].set_title('Предсказанная глубина, м'); ax[1].axis('off')
plt.tight_layout()

In [ ]:
# Сводная таблица: медиана предсказанной глубины в центральной области против dist
# Это предварительный индикатор, не финальная метрика: ориентация и точка лидара неизвестны.
import numpy as np, pandas as pd
labels=pd.read_csv(data_dir/'labels.csv')
records=[]
for _, row in labels.iterrows():
    depth_path=output_dir/(Path(row.image_filename).stem+'_depth.npy')
    if not depth_path.exists(): continue
    depth=np.load(depth_path)
    h,w=depth.shape
    y0,y1=int(.4*h),int(.6*h); x0,x1=int(.4*w),int(.6*w)
    roi=depth[y0:y1,x0:x1]
    valid=roi[np.isfinite(roi)&(roi>0)]
    pred=float(np.median(valid)) if len(valid) else np.nan
    records.append({**row.to_dict(),'center_roi_median_m':pred,
                    'abs_diff_m':abs(pred-float(row.lidar_dist_m)) if np.isfinite(pred) else np.nan})
comparison=pd.DataFrame(records)
comparison.to_csv('/kaggle/working/moge2_aerial_comparison.csv',index=False)
display(comparison.head(10))
print('CSV сохранён в /kaggle/working/moge2_aerial_comparison.csv')

In [ ]:
# Грубое сведение по полётам для первичного просмотра
comparison.groupby('flight').agg(
    frames=('image_filename','count'),
    median_model_depth_m=('center_roi_median_m','median'),
    median_lidar_dist_m=('lidar_dist_m','median'),
    median_absolute_difference_m=('abs_diff_m','median')
)

## Как читать результат

Цветная карта показывает оценку глубины в разных частях кадра; рядом с ней есть шкала в метрах. Числа в таблице сравнивают медиану центральной области с ближайшим `dist`, но лидара может смотреть не точно в центр кадра. Поэтому сначала смотрим, похожа ли карта на сцену и меняется ли расстояние по полёту правдоподобно. Не используем эти числа как подтверждение точности без калибровки камеры и лидара.

Файлы можно скачать из панели Output Kaggle: `moge2_aerial_output/` и `moge2_aerial_comparison.csv`.